# Understanding flight disruptions in the United States: A temporal and geospatial analysis

#### URBAN5123 Programming Tools for Urban Analytics 2024/25
#### 3061039

In [4]:
import pandas as pd
import geopandas as gpd
import hvplot.pandas

import calendar
from bokeh.models import DatetimeTickFormatter, HoverTool, NumeralTickFormatter
from IPython.display import display

## 1. Introduction

### 1.1. Context

Over the past few decades, flight data has provided valuable insights beyond the aviation industry, revealing broader societal and economic trends. Given the unpredictability of global events and the volatility of market conditions, a comprehensive analysis of existing flight data is crucial for anticipating the future of air travel. Sismanidou et al. (2022) and Sun et al. (2023) emphasise the importance of a data-driven, quantitative approach in fostering innovation and recovery in aviation, particularly in the wake of the COVID-19 pandemic. Building on this perspective, our research aims to examine how the volume and composition of air travel vary across both temporal and geospatial dimensions.

### 1.2. Research questions

This research investigates the temporal and geospatial distribution of domestic flight operations across the United States over the past two decades. For this research, we define "*disruptions*" to encompass three scenarios: delays, cancellations, and diversions. Our research questions can be split based on the type of analysis:

1. **Temporal analysis**: 
    - How has the number of domestic flights in the U.S. changed over time?
    - How have the rate of flight disruptions changed over time?
    - Have the primary causes of flight delays changed over time?
    - What is the influence of seasonality on the number of flights, rate of disruptions, and cause of delays?
    - What are the carriers and airports with the highest disruption rates over time?
<br />
2. **Geospatial analysis**:
    - Where are the airports with the highest rates of flight disruptions located?
    - Are there any airports or regions that are more prone to a specific delay cause? Where are they located?

Ultimately, this research aims to provide insights that can inform strategies for improving airport operations, minimising delays, and enhancing passenger experiences in the United States, especially in areas disproportionately impacted by flight disruptions.

### 1.3. Methods

The `pandas` library is used in this analaysis for data loading, cleaning, exploration, and transformation. `GeoPandas`, an extension of `pandas`, is used to handle data with location information (latitude and longitude). For both temporal and geospatial analysis, `hvplot` works with `pandas` and `GeoPandas` to create interactive temporal and geospatial visualisations. Specifically, line plots are used to showcase trends over time, stacked bar plots are used to show the breakdown of delay causes, and maps are created to show the location and severity of flight disruptions.

Other tools used in this analysis include the `calendar` module to format dates, the `Bokeh` library to enhance data visualisation formatting, and  the `IPython.display` module to render and display objects directly in the notebook output.

## 2. Data

### 2.1. Data sources

This research uses data from the US Department of Transportation’s 'Airline On-Time Statistics and Delay Causes’ dataset (Bureau of Transportation Statistics, 2025). This dataset provides detailed monthly information on the performance of domestic flights from June 2003 to December 2024, comprising the frequency and attributes of on-time flights, delays, cancellations and diversions. Flight delays are grouped into five broad categories:

1. **National Aviation System (NAS) delay**: a broad set of conditions from the national aviation system (e.g. heavy traffic volume, air traffic control).
2. **Security delay**: due to security issues at the airport (e.g. evacuation of terminal, long screening times).
3. **Extreme weather**: significant meteoreological conditions impeding flight operations (e.g. hurricane, blizzard).
4. **Late-arriving aircraft from a previous flight**: previous flight operating on the same aircraft arrived late.
5. **Air carrier delay**: circumstances within the airline carrier's control (e.g. maintenance or crew problems, baggage loading, fueling).


A flight is considered 'on time' if it arrives within 15 minutes of its scheduled arrival time, as recorded in the carrier's Computerized Reservations Systems (CRS). Otherwise, the flight is considered delayed. Arrival performance is evaluated based on the  gate arrival time, while departure performance is determined by the gate departure time. The causes of flight cancellations and diversions are not provided in the dataset.

After downloading the raw data as a ZIP file from the Bureau of Transportation Statistics (2025), we extract the `Airline_Delay_Cause.csv` file and read it into this notebook. 

In [11]:
df = pd.read_csv('./ot_delaycause1_DL/Airline_Delay_Cause.csv')

The 'Airline On-Time Statistics and Delay Causes' dataset provides the airport name and unique alpha-numeric code, but does not contain geographic coordinates. Therefore, we load a database containing the latitude and longitude of all airports worldwide into a dictionary (Borsetti, 2025). We will merge this location data with our main dataset in Section 2.3.4.

In [13]:
import airportsdata
airports_database = airportsdata.load('IATA')  # key is the IATA location code

### 2.2. Variable definitions

We first take a look at what variables are present in our dataset. Performance indicators (i.e. the columns/variables) are described in the `Download_Column_Definitions.xlsx` file.

In [15]:
definitions = pd.read_excel('./ot_delaycause1_DL/Download_Column_Definitions.xlsx')

In [16]:
pd.set_option('display.max_colwidth', None)
definitions.style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_table_styles([{'selector': 'th',  'props': [('text-align', 'left')]}])

Column Name,Column Definition
year,YYYY format
month,MM format (1-12)
carrier,Code assigned by US DOT to identify a unique airline carrier.
carrier_name,"Unique airline (carrier) is defined as one holding and reporting under the same DOT certificate regardless of its Code, Name, or holding company/corporation."
airport,A three character alpha-numeric code issued by the U.S. Department of Transportation which is the official designation of the airport.
airport_name,a place from which aircraft operate that usually has paved runways and maintenance facilities and often serves as a terminal
arr_flights,Arrival Flights
arr_del15,"Arrival Delay Indicator, 15 Minutes or More Arrival delay equals the difference of the actual arrival time minus the scheduled arrival time. A flight is considered on-time when it arrives less than 15 minutes after its published arrival time."
carrier_ct,Carrier Count for airline cause of delay
weather_ct,Weather Count for airline cause of delay


The sum of the counts for the five delay causes (`carrier_ct`, `weather_ct`, `nas_ct`, `security_ct`, `late_aircraft_ct`) equals `arr_del15`.

The sum of the number of minutes delayed for each cause (`carrier_delay`, `weather_delay`, `nas_delay`, `security_delay`, `late_aircraft_delay`) equals `arr_delay`.

### 2.3. Data cleaning

#### *2.3.1. Missing values*

We can look up the summary of our DataFrame by calling `df.info()` in pandas.

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396317 entries, 0 to 396316
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   year                 396317 non-null  int64  
 1   month                396317 non-null  int64  
 2   carrier              396317 non-null  object 
 3   carrier_name         396317 non-null  object 
 4   airport              396317 non-null  object 
 5   airport_name         396317 non-null  object 
 6   arr_flights          395660 non-null  float64
 7   arr_del15            395367 non-null  float64
 8   carrier_ct           395660 non-null  float64
 9   weather_ct           395660 non-null  float64
 10  nas_ct               395660 non-null  float64
 11  security_ct          395660 non-null  float64
 12  late_aircraft_ct     395660 non-null  float64
 13  arr_cancelled        395660 non-null  float64
 14  arr_diverted         395660 non-null  float64
 15  arr_delay        

657 to 950 observations have missing values for the count of flights. We take a look at these observations.

In [22]:
pd.set_option('display.max_columns', None) # show all columns
pd.reset_option('display.max_rows') # truncate rows
df[df['arr_del15'].isna()]

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
1292,2024,12,C5,CommuteAir LLC dba CommuteAir,AUS,"Austin, TX: Austin - Bergstrom International",1.0,NaN,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1590,2024,12,G4,Allegiant Air,BTV,"Burlington, VT: Burlington International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2305,2024,11,C5,CommuteAir LLC dba CommuteAir,DAY,"Dayton, OH: James M Cox/Dayton International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4983,2024,10,C5,CommuteAir LLC dba CommuteAir,PVD,"Providence, RI: Rhode Island Tf Green International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6455,2024,9,G7,GoJet Airlines LLC d/b/a United Express,HHH,"Hilton Head, SC: Hilton Head Airport",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392342,2003,9,RU,ExpressJet Airlines Inc.,TPA,"Tampa, FL: Tampa International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
392348,2003,9,TZ,ATA Airlines d/b/a ATA,ABQ,"Albuquerque, NM: Albuquerque International Sunport",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395156,2003,6,EV,Atlantic Southeast Airlines,ORF,"Norfolk, VA: Norfolk International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395170,2003,6,EV,Atlantic Southeast Airlines,SWF,"Newburgh/Poughkeepsie, NY: New York Stewart International",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


It appears that missing values occur when carriers have no recorded arrivals for a given month at a specific airport. To ensure consistency in the dataset, we replace missing values with zero, resulting in an identical non-null count for each column.

In [24]:
columns_to_fill = [
    'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 
    'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 
    'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 
    'late_aircraft_delay'
]

# Fill NaN with 0 for the specified columns
df[columns_to_fill] = df[columns_to_fill].fillna(0)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396317 entries, 0 to 396316
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   year                 396317 non-null  int64  
 1   month                396317 non-null  int64  
 2   carrier              396317 non-null  object 
 3   carrier_name         396317 non-null  object 
 4   airport              396317 non-null  object 
 5   airport_name         396317 non-null  object 
 6   arr_flights          396317 non-null  float64
 7   arr_del15            396317 non-null  float64
 8   carrier_ct           396317 non-null  float64
 9   weather_ct           396317 non-null  float64
 10  nas_ct               396317 non-null  float64
 11  security_ct          396317 non-null  float64
 12  late_aircraft_ct     396317 non-null  float64
 13  arr_cancelled        396317 non-null  float64
 14  arr_diverted         396317 non-null  float64
 15  arr_delay        

#### *2.3.2. Date-time transformation*

Currently, the year and month variables in our dataset are stored in separate columns. To streamline time-based analysis, we combine these two variables into a new `year_month` column.

In [27]:
df['year_month'] = df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)
df['year_month'] = df['year_month'].astype(str)

#### *2.3.3. Data type transformation*

We also convert the carrier and airport variables to categorical variables.  This step not only simplifies the computation of statistics and the visualisation of patterns in subsequent sections, but also reduces overall memory usage.

In [29]:
cat_cols = ['carrier', 'carrier_name', 'airport', 'airport_name']
for c in cat_cols:
    df[c] = df[c].astype('category')

Next, we convert count variables from floating-point numbers to integers. When multiple causes are assigned to one delayed flight, each cause is prorated based on the delayed minutes it is responsible for, which explains why count variables sorted by cause of delay have non-integer values. We keep these variables in their original data format.

In [31]:
int_cols = [
    'arr_flights', 'arr_del15', 'arr_cancelled', 'arr_diverted', 
    'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 
    'late_aircraft_delay'
]

for c in int_cols:
    df[c] = df[c].astype('int64')

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396317 entries, 0 to 396316
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   year                 396317 non-null  int64   
 1   month                396317 non-null  int64   
 2   carrier              396317 non-null  category
 3   carrier_name         396317 non-null  category
 4   airport              396317 non-null  category
 5   airport_name         396317 non-null  category
 6   arr_flights          396317 non-null  int64   
 7   arr_del15            396317 non-null  int64   
 8   carrier_ct           396317 non-null  float64 
 9   weather_ct           396317 non-null  float64 
 10  nas_ct               396317 non-null  float64 
 11  security_ct          396317 non-null  float64 
 12  late_aircraft_ct     396317 non-null  float64 
 13  arr_cancelled        396317 non-null  int64   
 14  arr_diverted         396317 non-null  int64   
 15  

#### *2.3.4. Location information*

To perform geospatial analysis, we need to append the geographic coordinates corresponding to each airport in our dataset. We convert the previously loaded `airports_database` into a DataFrame and merge it with our main DataFrame.

In [34]:
# Convert the dictionary to a DataFrame
airports_database_df = pd.DataFrame.from_dict(airports_database, orient='index', columns=['lat', 'lon'])
airports_database_df.reset_index(inplace=True)
airports_database_df.rename(columns={'index': 'airport'}, inplace=True)

# Merge the two DataFrames on the 'airport' column
df = pd.merge(df, airports_database_df, on='airport', how='left')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396317 entries, 0 to 396316
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   year                 396317 non-null  int64   
 1   month                396317 non-null  int64   
 2   carrier              396317 non-null  category
 3   carrier_name         396317 non-null  category
 4   airport              396317 non-null  object  
 5   airport_name         396317 non-null  category
 6   arr_flights          396317 non-null  int64   
 7   arr_del15            396317 non-null  int64   
 8   carrier_ct           396317 non-null  float64 
 9   weather_ct           396317 non-null  float64 
 10  nas_ct               396317 non-null  float64 
 11  security_ct          396317 non-null  float64 
 12  late_aircraft_ct     396317 non-null  float64 
 13  arr_cancelled        396317 non-null  int64   
 14  arr_diverted         396317 non-null  int64   
 15  

It appears that there are 348 observations without latitude and longitude information. Let us take a look at these observations.

In [36]:
temp_df = df[df['lat'].isna()]
display(temp_df)
temp_df['airport'].unique()

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,year_month,lat,lon
117593,2019,10,AX,Trans States Airlines,ISN,"Williston, ND: Sloulin Field International",34,6,1.83,0.00,1.00,0.0,3.17,0,0,762,275,0,22,0,465,2019-10,NaN,NaN
118724,2019,10,OO,SkyWest Airlines Inc.,ISN,"Williston, ND: Sloulin Field International",17,2,0.00,0.00,0.00,0.0,2.00,0,1,118,0,0,0,0,118,2019-10,NaN,NaN
119752,2019,9,AX,Trans States Airlines,ISN,"Williston, ND: Sloulin Field International",115,14,6.80,0.00,0.41,0.0,6.80,0,0,912,385,0,24,0,503,2019-09,NaN,NaN
120855,2019,9,OO,SkyWest Airlines Inc.,ISN,"Williston, ND: Sloulin Field International",60,3,2.00,0.00,1.00,0.0,0.00,0,2,941,681,0,260,0,0,2019-09,NaN,NaN
122539,2019,8,OO,SkyWest Airlines Inc.,ISN,"Williston, ND: Sloulin Field International",88,9,2.00,1.22,2.31,0.0,3.47,0,0,1100,745,94,69,0,192,2019-08,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393456,2003,8,OO,SkyWest Airlines Inc.,PFN,"Panama City, FL: Bay County",15,1,0.00,0.00,1.00,0.0,0.00,0,0,49,0,0,49,0,0,2003-08,NaN,NaN
394321,2003,7,EV,Atlantic Southeast Airlines,PFN,"Panama City, FL: Bay County",257,132,51.01,8.84,51.80,0.0,20.34,6,2,6415,2733,748,1884,0,1050,2003-07,NaN,NaN
394703,2003,7,OO,SkyWest Airlines Inc.,PFN,"Panama City, FL: Bay County",12,1,0.00,1.00,0.00,0.0,0.00,0,0,130,0,130,0,0,0,2003-07,NaN,NaN
395157,2003,6,EV,Atlantic Southeast Airlines,PFN,"Panama City, FL: Bay County",250,114,46.29,6.75,33.89,0.6,26.47,1,0,5193,2222,579,1186,12,1194,2003-06,NaN,NaN


array(['ISN', 'TKI', 'PFN'], dtype=object)

There are 3 airports lacking coordinate information. We can fill in these values manually.

In [38]:
df.loc[df['airport'] == 'ISN', 'lat'] = 48.17806
df.loc[df['airport'] == 'ISN', 'lon'] = -103.64222

df.loc[df['airport'] == 'TKI', 'lat'] = 33.17806
df.loc[df['airport'] == 'TKI', 'lon'] = -96.59056

df.loc[df['airport'] == 'PFN', 'lat'] = 30.21222
df.loc[df['airport'] == 'PFN', 'lon'] = -85.68278

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396317 entries, 0 to 396316
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   year                 396317 non-null  int64   
 1   month                396317 non-null  int64   
 2   carrier              396317 non-null  category
 3   carrier_name         396317 non-null  category
 4   airport              396317 non-null  object  
 5   airport_name         396317 non-null  category
 6   arr_flights          396317 non-null  int64   
 7   arr_del15            396317 non-null  int64   
 8   carrier_ct           396317 non-null  float64 
 9   weather_ct           396317 non-null  float64 
 10  nas_ct               396317 non-null  float64 
 11  security_ct          396317 non-null  float64 
 12  late_aircraft_ct     396317 non-null  float64 
 13  arr_cancelled        396317 non-null  int64   
 14  arr_diverted         396317 non-null  int64   
 15  

### 2.4. Summary statistics

#### *2.4.1. Observation-level statistics*

Now that we have completed data loading and cleaning, we can preview our updated DataFrame. Each observation in our dataset represents an airline carrier's performance for a given month at a given airport.

In [43]:
pd.set_option('display.max_columns', None) # show all columns
pd.reset_option('display.max_rows') # truncate rows
df

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,year_month,lat,lon
0,2024,12,MQ,Envoy Air,EVV,"Evansville, IN: Evansville Regional",61,9,1.52,1.08,0.56,0.0,5.84,0,0,732,47,90,19,0,576,2024-12,38.04081,-87.52850
1,2024,12,MQ,Envoy Air,EWR,"Newark, NJ: Newark Liberty International",107,42,6.01,5.89,25.16,0.0,4.94,0,0,2531,335,491,1251,0,454,2024-12,40.69248,-74.16869
2,2024,12,MQ,Envoy Air,EYW,"Key West, FL: Key West International",169,31,3.37,0.71,11.44,0.0,15.48,5,3,1596,143,52,468,0,933,2024-12,24.55612,-81.75996
3,2024,12,MQ,Envoy Air,FAR,"Fargo, ND: Hector International",171,35,4.64,2.12,15.32,0.0,12.92,2,0,2428,245,184,575,0,1424,2024-12,46.92064,-96.81575
4,2024,12,MQ,Envoy Air,FSD,"Sioux Falls, SD: Joe Foss Field",69,14,2.00,2.47,4.70,0.0,4.83,1,0,720,86,154,191,0,289,2024-12,43.58202,-96.74192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
396312,2003,6,DL,Delta Air Lines Inc.,SEA,"Seattle, WA: Seattle/Tacoma International",480,84,25.69,3.09,33.96,0.0,21.26,0,0,3744,978,164,1023,0,1579,2003-06,47.44989,-122.31178
396313,2003,6,DL,Delta Air Lines Inc.,SFO,"San Francisco, CA: San Francisco International",505,111,21.78,2.24,73.81,0.0,13.17,3,0,4284,1376,138,2132,0,638,2003-06,37.61881,-122.37542
396314,2003,6,DL,Delta Air Lines Inc.,SJC,"San Jose, CA: Norman Y. Mineta San Jose International",146,36,6.99,0.00,26.85,0.0,2.15,0,0,896,205,0,607,0,84,2003-06,37.36299,-121.92862
396315,2003,6,DL,Delta Air Lines Inc.,SJU,"San Juan, PR: Luis Munoz Marin International",95,13,3.66,0.00,7.92,0.0,1.42,0,0,367,120,0,210,0,37,2003-06,18.43940,-66.00213


In [44]:
print("Number of years =", df['year'].nunique())

Number of years = 22


From the output generated, we can see that there are 396,317 observations across 22 years of monthly data. Some values do not necessarily add up to the total as they are rounded.

#### *2.4.2. Carrier-level statistics*

Next, we note how many airline carriers are represented in our dataset through the unique airline code assigned by the US Department of Transportation.

In [47]:
print("Number of unique carriers =", df['carrier'].nunique())

Number of unique carriers = 38


A total of 38 unique airline carriers are represented in our dataset spanning from June 2003 to December 2024. 

To gauge the completeness of the dataset, we can explore the number of carriers that have contributed data for each month during this time span and construct a line plot.

In [49]:
df.groupby('year_month', observed=True)['carrier'].nunique()

year_month
2003-06    17
2003-07    17
2003-08    17
2003-09    17
2003-10    17
           ..
2024-08    21
2024-09    21
2024-10    21
2024-11    21
2024-12    21
Name: carrier, Length: 259, dtype: int64

In [50]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Group by year_month and count unique carriers
unique_carriers_per_month = df.groupby('year_month', observed=True)['carrier'].nunique()

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Resetting index to use for plotting
unique_carriers_df = unique_carriers_per_month.reset_index()
unique_carriers_df.columns = ['year_month', 'carrier_count']  # Renaming columns for clarity

# Set up DatetimeTickFormatter for customized tick labels
tickfmt = DatetimeTickFormatter(years="%Y-%m", months="%Y-%m")

# Set up tooltips for interactivity
tooltips = [
    ("Year-Month", "@year_month{%Y-%m}"),  # Formatting date as Year-Month
    ("Carrier Count", "@carrier_count"),
]
hover = HoverTool(tooltips=tooltips, formatters={"@year_month": "datetime"})

# Plot using hvplot
unique_carriers_df.hvplot(
    x='year_month', 
    y='carrier_count', 
    kind='line', 
    xformatter=tickfmt, 
    tools=[hover], 
    title="Figure 1: Number of unique carriers reported per month",
    ylabel="Number of carriers"
)

:Curve   [year_month]   (carrier_count)

It appears that the monthly reported number of unique carriers ranges from 12 to 28. Notably, there was a sharp increase in the number of unique carriers reporting flight statistics between December 2017 and Jaunary 2018.

#### *2.4.3. Airport-level statistics*

We also note how many airports are represented in our dataset through the alpha-numeric code of each airport.

In [53]:
print("Number of unique airports =", df['airport'].nunique())

Number of unique airports = 425


A total of 425 unique airports are represented in our dataset spanning from June 2003 to December 2024. We can plot a map to examine the geographic distribution of these airports across the United States.

In [55]:
# Prepare the data: only keep unique airports with their lat/lon
airports = df[['airport', 'lat', 'lon']].drop_duplicates()

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(airports, geometry=gpd.points_from_xy(airports['lon'], airports['lat']))

# Rename columns for better hover display
gdf = gdf.rename(columns={
    'airport': 'Airport'})

# Plot the geographic distribution of airports on a map using hvplot
gdf.hvplot(
    geo=True,                  # Enable geographic plotting
    x='Longitude',             # Longitude for x-axis
    y='Latitude',              # Latitude for y-axis
    kind='points',             # Use points to plot airport locations
    hover_cols=['Longitude', 'Latitude', 'Airport'],    # Display airport names on hover
    tiles='OSM',               # Use OpenStreetMap tiles for the background map
    title="Figure 2: Geographic distribution of reported airports",
    size=5,                     # Adjust the point size for visibility
    xlim=(-170, -60),           # Longitude bounds for US view
    ylim=(15, 72)               # Latitude bounds for US view
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Airport)

By dragging across the map, we can observe that our dataset includes airports not only in the contiguous United States, but also in Alaska, Hawaii, and various U.S. territories (Guam and American Samoa in the Pacific, and Puerto Rico in the Caribbean).

We can also explore the number of airports that have contributed data for each month during this time span.

In [57]:
df.groupby('year_month', observed=True)['airport'].nunique()

year_month
2003-06    275
2003-07    274
2003-08    272
2003-09    272
2003-10    268
          ... 
2024-08    354
2024-09    352
2024-10    352
2024-11    349
2024-12    352
Name: airport, Length: 259, dtype: int64

In [58]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Group by year_month and count unique airports
unique_airports_per_month = df.groupby('year_month', observed=True)['airport'].nunique()

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Resetting index to use for plotting
unique_carriers_df = unique_airports_per_month.reset_index()
unique_carriers_df.columns = ['year_month', 'airport_count']  # Renaming columns for clarity

# Set up DatetimeTickFormatter for customized tick labels
tickfmt = DatetimeTickFormatter(years="%Y-%m", months="%Y-%m")

# Set up tooltips for interactivity
tooltips = [
    ("Year-Month", "@year_month{%Y-%m}"),  # Formatting date as Year-Month
    ("Airport Count", "@airport_count"),
]
hover = HoverTool(tooltips=tooltips, formatters={"@year_month": "datetime"})

# Plot using hvplot
unique_carriers_df.hvplot(
    x='year_month', 
    y='airport_count', 
    kind='line', 
    xformatter=tickfmt, 
    tools=[hover], 
    title="Figure 3: Number of unique airports reported per month",
    ylabel="Number of airports"
)

:Curve   [year_month]   (airport_count)

The monthly reported number of unique airports ranges from 268 to 374. Similar to the trend observed for carriers in Figure 1, there was a sharp increase in the number of airlines reporting data between December 2017 and January 2018.

## 3. Temporal analysis

### 3.1. Long-term trends

We start by examining the long-term trend in the total number of flights reported across all carriers and airports between June 2003 and December 2024.

In [62]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Group overall metrics by year_month:
agg_metrics = df.groupby('year_month', observed=True).agg({
    'arr_flights': 'sum'        # Total flights per month
}).reset_index()

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Create a DataFrame with the relevant columns for plotting
rate_df = agg_metrics[['year_month', 'arr_flights']]

# Set up DatetimeTickFormatter for customized tick labels on the x-axis
tickfmt = DatetimeTickFormatter(years="%Y-%m", months="%Y-%m")

# Set up tooltips for interactivity
tooltips = [
    ("Year-Month", "@year_month{%Y-%m}"),
    ("Flights", "@arr_flights{0,0}")
]
hover = HoverTool(tooltips=tooltips, formatters={"@year_month": "datetime"})

# Plot using hvplot with multiple y columns (each becomes a line)
rate_df.hvplot(
    x='year_month', 
    y='arr_flights', 
    kind='line', 
    xformatter=tickfmt, 
    tools=[hover], 
    title="Figure 4: Overall flights per month",
    ylabel="Number of flights"
).opts(height=400, yformatter=NumeralTickFormatter(format='0,0'))

:Curve   [year_month]   (arr_flights)

Monthly domestic flights in the U.S. grew from June 2003 (536,496) to July 2008 (653,279). Thereafter, a decreasing trend was observed until November 2017 (454,162). This downturn can largely be attributed to the aftermath of the Great Recession, which occurred between late 2007 and early 2009. The International Air Transport Association (IATA) described the recession as "the deepest [downturn] experienced by the commercial airline industry since the 1930s" (Segal, 2018). Although the recession officially lasted a few years, its effects on the aviation industry persisted well into the 2010s, contributing to a prolonged period of reduced air traffic. 

There was a sharp rise in the monthly domestic flights from December 2017 (464,205) to January 2018 (621,398). This is in line with our findings from Figures 1 and 3. This increase in reported flight numbers is likely due to technical factors rather than a genuine rise, possibly stemming from data adjustments or changes in reporting methodology.

The COVID-19 pandemic had an unprecendented impact on domestic aviation. A sharp drop in the number of monthly domestic flights was recorded between April 2020 (331,238) and May 2021 (520,059). Monthly domestic flights steadily increased from June 2021 (573,779) to December 2024 (631,944) as vacinnes became more widely available across the U.S, allowing air travel to gradually recover. Nevertheless, overall flight traffic has not returned to pre-pandemic levels.

From Figure 5, we can also observe how air travel exhibits strong seasonality, flucutating between peak travel months and low travel months This which will be explored in greater detail in Section 3.2.

We now look at how overall rates of delays, cancellations, and diversions have changed month-on-month from June 2003 to December 2024.

In [64]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Group overall metrics by year_month:
agg_metrics = df.groupby('year_month', observed=True).agg({
    'arr_del15': 'sum',        # Total delayed flights per month
    'arr_cancelled': 'sum',    # Total cancelled flights per month
    'arr_diverted': 'sum',     # Total diverted flights per month
    'arr_flights': 'sum'       # Total arrived flights per month
}).reset_index()

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Compute rates for each month
agg_metrics['delay_rate'] = agg_metrics['arr_del15'] / agg_metrics['arr_flights']
agg_metrics['cancellation_rate'] = agg_metrics['arr_cancelled'] / agg_metrics['arr_flights']
agg_metrics['diversion_rate'] = agg_metrics['arr_diverted'] / agg_metrics['arr_flights']

# Create a DataFrame with the relevant columns for plotting
rate_df = agg_metrics[['year_month', 'delay_rate', 'cancellation_rate', 'diversion_rate']]

# Rename columns for custom legend labels
rate_df_renamed = rate_df.rename(columns={
    'delay_rate': 'Delay Rate',
    'cancellation_rate': 'Cancellation Rate',
    'diversion_rate': 'Diversion Rate'
})

# Set up DatetimeTickFormatter for customized tick labels on the x-axis
tickfmt = DatetimeTickFormatter(years="%Y-%m", months="%Y-%m")

# Set up tooltips for interactivity
tooltips = [
    ("Year-Month", "@year_month{%Y-%m}"),
    ("Delay Rate", "@{Delay Rate}{0.000}"),
    ("Cancellation Rate", "@{Cancellation Rate}{0.000}"),
    ("Diversion Rate", "@{Diversion Rate}{0.000}")
]
hover = HoverTool(tooltips=tooltips, formatters={"@year_month": "datetime"})

# Plot using hvplot with renamed columns for the legend
rate_df_renamed.hvplot(
    x='year_month', 
    y=['Delay Rate', 'Cancellation Rate', 'Diversion Rate'], 
    kind='line', 
    xformatter=tickfmt, 
    tools=[hover], 
    title="Figure 5: Overall delay, cancellation, and diversion rates per month",
    xlabel="Year-Month",
    ylabel="Rate"
).opts(legend_position='bottom', height=400)

:NdOverlay   [Variable]
   :Curve   [year_month]   (value)

A comparison of the different rates in Figure 5 shows that flight delays are the most common type of disruption, followed by cancellations, while diversions occur the least frequently. The only time when the overall cancellation rate was higher than the overall delay rate was in April 2020, coinciding with the start of COVID-19 travel restrictions in the U.S. (Atwood *et al.*, 2020). In that month, 41 flights were cancelled for every 100 flight arrivals.

Similar to the overall flight numbers in Figure 4, flight disruptions also exhibit a degree of seasonality. This will be explored in Section 3.2.

### 3.2. Seasonality

To account for seasonality, we calculate the total number of flights for each month over the available years, take the median value for each month (January to December), and then plot the result. This helps us understand recurring seasonal patterns across the years. Given the unprecedented impact of COVID-19 on air travel as highlighted in Figures 4 and 5, we exclude the monthly data from April 2020 to May 2021 from our analysis. This allows us to have a better understanding of usual operations.

In [67]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Group by month, calculate the total number of flights for each month across all years
monthly_total_flights = df_filtered.groupby('month')['arr_flights'].sum().reset_index()

# Count the number of available years for each month (excluding missing months like COVID-19 impacted months)
years_per_month = df_filtered.groupby('month')['year'].nunique().reset_index()
years_per_month.columns = ['month', 'available_years']

# Merge total flights with the count of available years
monthly_avg_flights = pd.merge(monthly_total_flights, years_per_month, on='month')

# Calculate the mean number of flights per month (i.e. total flights divided by number of years)
monthly_avg_flights['mean_flights'] = monthly_avg_flights['arr_flights'] / monthly_avg_flights['available_years']

# Convert month numbers (1-12) to month names
monthly_avg_flights['Month Name'] = monthly_avg_flights['month'].apply(lambda x: calendar.month_abbr[x])

# Set up tooltips for interactivity
tooltips = [
    ("Month", "@{Month Name}"),
    ("Mean number of flights", "@mean_flights{0,}")
]
hover = HoverTool(tooltips=tooltips)

# Plot using hvplot
monthly_avg_flights.hvplot(
    x='Month Name', 
    y='mean_flights', 
    kind='line', 
    tools=[hover],
    title='Figure 6: Mean number of flights from January to December \n(excluding COVID-19 period)',
    xlabel='Month', 
    ylabel='Mean number of flights'
).opts(height=400, yformatter=NumeralTickFormatter(format='0,0'))

:Curve   [Month Name]   (mean_flights)

From Figure 6, we can see that the peak travel months are the summer months of July (596,967 flights on average) and August (592,954 flights on average), while February records the lowest average number of flights (510,157). This seasonal pattern highlights significant fluctuations in air travel demand throughout the year, reflecting established travel trends driven by vacation periods and weather conditions.

Next, we see how flight delays, cancellations, and diversions are affected by seasonality, while excluding the COVID-19 period.

In [69]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Group by month and calculate total delays, cancellations, and diversions for each month across all years
monthly_total_metrics = df_filtered.groupby('month').agg({
    'arr_del15': 'sum',         # Total delays per month
    'arr_cancelled': 'sum',     # Total cancellations per month
    'arr_diverted': 'sum',      # Total diversions per month
    'arr_flights': 'sum'        # Total flights per month
}).reset_index()

# Count the number of available years for each month (excluding COVID-19 impacted months)
years_per_month = df_filtered.groupby('month')['year'].nunique().reset_index()
years_per_month.columns = ['month', 'available_years']

# Merge total metrics with the count of available years
monthly_avg_metrics = pd.merge(monthly_total_metrics, years_per_month, on='month')

# Calculate the mean rates for delay, cancellation, and diversion
monthly_avg_metrics['delay_rate'] = monthly_avg_metrics['arr_del15'] / monthly_avg_metrics['arr_flights']
monthly_avg_metrics['cancellation_rate'] = monthly_avg_metrics['arr_cancelled'] / monthly_avg_metrics['arr_flights']
monthly_avg_metrics['diversion_rate'] = monthly_avg_metrics['arr_diverted'] / monthly_avg_metrics['arr_flights']

# Convert month numbers (1-12) to month names
monthly_avg_metrics['Month Name'] = monthly_avg_metrics['month'].apply(lambda x: calendar.month_abbr[x])

# Rename the columns for better legend labels
monthly_avg_metrics_renamed = monthly_avg_metrics.rename(columns={
    'delay_rate': 'Delay Rate',
    'cancellation_rate': 'Cancellation Rate',
    'diversion_rate': 'Diversion Rate'
})

# Set up tooltips for interactivity
tooltips = [
    ("Month", "@{Month Name}"),
    ("Delay Rate", "@{Delay Rate}{0.000}"),
    ("Cancellation Rate", "@{Cancellation Rate}{0.000}"),
    ("Diversion Rate", "@{Diversion Rate}{0.000}")
]
hover = HoverTool(tooltips=tooltips)

# Plot using hvplot for all three metrics
monthly_avg_metrics_renamed.hvplot(
    x='Month Name', 
    y=['Delay Rate', 'Cancellation Rate', 'Diversion Rate'], 
    kind='line', 
    tools=[hover],
    title='Figure 7: Mean delay, cancellation, and diversion rates from January to December \n(excluding COVID-19 period)',
    xlabel='Month', 
    ylabel='Rate'
).opts(legend_position='bottom', height=400)

:NdOverlay   [Variable]
   :Curve   [Month Name]   (value)

While cancellation and diversion rates remain relatively stable throughout the year, delay rates peak during the summer months of June and July, with approximately 23% of all flights experiencing delays. In contrast, September sees the lowest delay rates, averaging 15%.

### 3.3. Disruptions by carrier

We now move on to investigate which carrier is most prone to delays, cancellations, and diversions over the time frame of our dataset.

In [72]:
# Find the carrier most prone to delays, cancellations and diversions month-on-month (delay/cancellation/diversion count / arrival count)

# Group by carrier and month, then sum the delay count and arrival count
grouped_df = df.groupby(['carrier', 'year_month'], observed=True).agg({
    'arr_del15': 'sum',  # Sum of delays per month per carrier
    'arr_cancelled': 'sum', # Sum of cancellations per month per carrier
    'arr_diverted': 'sum', # Sum of diversions per month per carrier
    'arr_flights': 'sum'  # Sum of arrivals per month per carrier
}).reset_index()

# Compute the ratio (delay/cancellation/diversion_count / arrival_count)
grouped_df['delay_ratio'] = grouped_df['arr_del15'] / grouped_df['arr_flights']
grouped_df['cancel_ratio'] = grouped_df['arr_cancelled'] / grouped_df['arr_flights']
grouped_df['divert_ratio'] = grouped_df['arr_diverted'] / grouped_df['arr_flights']

# Find the carrier with the highest delay, cancellation, and diversion ratio per year_month
top_delayed_carrier = grouped_df.loc[grouped_df.groupby('year_month', observed=True)['delay_ratio'].idxmax()]
top_cancelled_carrier = grouped_df.loc[grouped_df.groupby('year_month', observed=True)['cancel_ratio'].idxmax()]
top_diverted_carrier = grouped_df.loc[grouped_df.groupby('year_month', observed=True)['divert_ratio'].idxmax()]

# Merge the results into one DataFrame
disruptions_by_carrier = top_delayed_carrier[['year_month', 'carrier']].rename(columns={'carrier': 'Top Delayed Carrier'})
disruptions_by_carrier['Top Cancelled Carrier'] = top_cancelled_carrier['carrier'].values
disruptions_by_carrier['Top Diverted Carrier'] = top_diverted_carrier['carrier'].values

# Display all rows (remove row limit)
pd.set_option('display.max_rows', None)

# Display the result with styling
disruptions_by_carrier.style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
])

year_month,Top Delayed Carrier,Top Cancelled Carrier,Top Diverted Carrier
2003-06,EV,DH,RU
2003-07,EV,DH,RU
2003-08,FL,DH,B6
2003-09,CO,MQ,AS
2003-10,AS,AS,AS
2003-11,MQ,MQ,AS
2003-12,FL,MQ,AS
2004-01,TZ,AS,AS
2004-02,FL,EV,AS
2004-03,HP,MQ,AS


In [73]:
disruptions_by_carrier['Top Delayed Carrier'].value_counts()

Top Delayed Carrier
B6    47
F9    38
EV    21
G4    15
VX    13
OH    13
NK    13
AA    12
C5    12
KS     9
FL     9
WN     8
MQ     7
AS     6
UA     6
HA     5
TZ     4
CO     3
US     3
NW     3
DH     2
AX     2
HP     2
OO     1
9E     1
YV     1
XE     1
G7     1
ZW     1
9K     0
AQ     0
DL     0
EM     0
CP     0
RU     0
QX     0
PT     0
YX     0
Name: count, dtype: int64

In [74]:
disruptions_by_carrier['Top Cancelled Carrier'].value_counts()

Top Cancelled Carrier
MQ    64
EV    29
OH    17
KS    17
YV    17
OO    11
NK    11
G7    10
G4     9
B6     8
F9     8
9E     7
EM     6
AA     5
HA     5
QX     5
AS     5
DH     3
DL     3
YX     3
C5     3
XE     2
ZW     2
PT     1
AQ     1
AX     1
VX     1
FL     1
CP     1
CO     1
RU     1
WN     1
9K     0
HP     0
TZ     0
NW     0
UA     0
US     0
Name: count, dtype: int64

In [75]:
disruptions_by_carrier['Top Diverted Carrier'].value_counts()

Top Diverted Carrier
AS    50
AA    38
B6    38
KS    18
OO    15
EV    12
CO     9
G4     8
PT     8
XE     8
RU     7
VX     7
UA     6
C5     6
EM     6
YV     5
QX     3
AX     2
WN     2
NW     2
FL     2
G7     2
MQ     1
DL     1
9E     1
OH     1
US     1
9K     0
AQ     0
CP     0
HA     0
HP     0
F9     0
DH     0
TZ     0
NK     0
YX     0
ZW     0
Name: count, dtype: int64

Between June 2003 and December 2024, JetBlue (B6) was most frequently ranked as the most delayed carrier, topping the list 47 times. It was followed by Frontier Airlines (F9), which held the top spot 38 times.

During the same period, Envoy Air (MQ) was most frequently ranked as the carrier with the highest cancellation rates, topping the ranking 64 times. It was followed by ExpressJet (EV), which topped the rainking 29 times.

For flight diversions, Alaska Airlines (AS) ranked highest 50 times, while American Airlines (AA) and JetBlue (B6) each topped the ranking 38 times.

### 3.4. Disruptions by airport

Next, we investigate which airport encounters the highest rate of delays, cancellations, and diversions per month.

In [78]:
# Find the airport most prone to delays, cancellations and diversions month-on-month (delay/cancellation/diversion count / arrival count)

# Group by airport and month, then sum the delay count and arrival count
grouped_df = df.groupby(['airport', 'year_month'], observed=True).agg({
    'arr_del15': 'sum',  # Sum of delays per month per airport
    'arr_cancelled': 'sum', # Sum of cancellations per month per airport
    'arr_diverted': 'sum', # Sum of diversions per month per airport
    'arr_flights': 'sum'  # Sum of arrivals per month per airport
}).reset_index()

# Compute the ratio (delay/cancellation/diversion_count / arrival_count)
grouped_df['delay_ratio'] = grouped_df['arr_del15'] / grouped_df['arr_flights']
grouped_df['cancel_ratio'] = grouped_df['arr_cancelled'] / grouped_df['arr_flights']
grouped_df['divert_ratio'] = grouped_df['arr_diverted'] / grouped_df['arr_flights']

# Find the airport with the highest delay, cancellation, and diversion ratio per year_month
top_delayed_airport = grouped_df.loc[grouped_df.groupby('year_month', observed=True)['delay_ratio'].idxmax()]
top_cancelled_airport = grouped_df.loc[grouped_df.groupby('year_month', observed=True)['cancel_ratio'].idxmax()]
top_diverted_airport = grouped_df.loc[grouped_df.groupby('year_month', observed=True)['divert_ratio'].idxmax()]

# Merge the results into one DataFrame
disruptions_by_airport = top_delayed_airport[['year_month', 'airport']].rename(columns={'airport': 'Top Delayed Airport'})
disruptions_by_airport['Top Cancelled Airport'] = top_cancelled_airport['airport'].values
disruptions_by_airport['Top Diverted Airport'] = top_diverted_airport['airport'].values

# Display all rows (remove row limit)
pd.set_option('display.max_rows', None)

# Display the result with styling
disruptions_by_airport.style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
])

year_month,Top Delayed Airport,Top Cancelled Airport,Top Diverted Airport
2003-06,PFN,DUT,CYS
2003-07,EGE,DUT,BFF
2003-08,EGE,DUT,GUC
2003-09,ACK,DUT,ADK
2003-10,ADK,DUT,KTN
2003-11,ADK,WRG,DLG
2003-12,MQT,DUT,WRG
2004-01,DBQ,DUT,MKK
2004-02,ADK,SCC,MKK
2004-03,DLG,ADQ,JNU


In [79]:
disruptions_by_airport['Top Delayed Airport'].value_counts()

Top Delayed Airport
ADK    21
PPG    19
OTH    19
GUM    12
MQT     7
DUT     6
CEC     6
PVU     5
GUC     5
EGE     5
SMX     5
EWR     4
BKG     4
OWB     4
ACK     4
PSE     4
FLO     3
DLG     3
MEI     3
AKN     3
SCK     3
OGD     3
HGR     3
TOL     3
SFO     3
ILG     3
TTN     3
ELM     2
DBQ     2
OAJ     2
WRG     2
RDD     2
SOP     2
SLE     2
ISO     2
PSG     2
YAK     2
PIR     2
ALO     2
CMX     2
PAH     2
BRW     2
DIK     2
BGR     2
SUX     2
UST     2
ACY     2
RKS     1
COD     1
OTZ     1
GST     1
BQN     1
HKY     1
CWA     1
OME     1
PFN     1
RHI     1
ROW     1
GNV     1
IPL     1
FAY     1
SIT     1
ILE     1
MCN     1
ABR     1
BPT     1
PMD     1
ABY     1
LAW     1
DAB     1
LWB     1
STC     1
PIE     1
SWF     1
CMI     1
JLN     1
HOB     1
MOD     1
LAN     1
MSP     1
MOT     1
PLN     1
HTS     1
LRD     1
CYS     1
CIC     1
HDN     1
CDV     1
YNG     1
TXK     1
ACV     1
RIW     1
USA     1
PQI     1
UIN     1
PGD     1
ATY     1
IFP     1


In [80]:
disruptions_by_airport['Top Cancelled Airport'].value_counts()

Top Cancelled Airport
ADK    32
MMH    19
ADQ    18
DUT    11
ALO     7
DLG     5
CMX     5
JHM     5
OAJ     4
ASE     4
PSE     4
PQI     4
DVL     4
RHI     3
SCE     3
OTH     3
CYS     3
ALW     3
UST     3
IAG     3
BPT     3
ACK     3
STC     3
LCH     3
OME     3
SCC     3
LAR     3
BET     2
MOD     2
ACV     2
ITH     2
LAW     2
TXK     2
BGM     2
SUN     2
DBQ     2
MEI     2
MSY     2
APN     2
AKN     2
BIH     2
TOL     2
GFK     2
DCA     1
HTS     1
HRL     1
IPL     1
INL     1
EAU     1
PLN     1
GTR     1
EKO     1
CSG     1
FLO     1
SBN     1
DHN     1
HKY     1
TTN     1
WRG     1
MQT     1
PNS     1
VIS     1
FLG     1
ABR     1
MKG     1
CIC     1
MBS     1
CEC     1
HOB     1
LAN     1
EGE     1
AZO     1
IYK     1
FSM     1
HVN     1
IPT     1
OWB     1
STT     1
CRP     1
MRY     1
GGG     1
ART     1
BFM     1
COU     1
BLI     1
BQN     1
ITO     1
RDD     1
LIH     1
OGS     1
PIR     1
ILG     1
BRW     1
SMX     1
DIK     1
EAT     1
DRT     1
RSW     

In [81]:
disruptions_by_airport['Top Diverted Airport'].value_counts()

Top Diverted Airport
SUN    54
OGD    11
OTH    11
BRW     8
CDC     8
OME     7
OTZ     7
YAK     7
COD     6
CYS     5
ADK     5
ASE     5
SMX     5
JNU     4
DUT     4
DVL     4
CEC     4
SCC     4
BTM     4
PSM     4
JMS     3
HGR     3
DLG     3
BFF     3
WYS     3
CDV     3
CMX     3
FMN     3
PUB     3
PSG     3
LBL     2
CDB     2
MEI     2
PSE     2
WRG     2
EGE     2
MKK     2
GST     2
MVY     2
ACV     2
ORH     2
PIR     2
PPG     2
ART     2
GUC     1
KTN     1
GRK     1
SUX     1
TEX     1
BLI     1
ERI     1
EFD     1
LBF     1
HHH     1
MTH     1
PVU     1
ADQ     1
HLN     1
FLO     1
AKN     1
HPN     1
BKG     1
HYS     1
OGS     1
PLN     1
UST     1
DIK     1
STC     1
HOB     1
DLH     1
SWF     1
ABR     1
BGM     1
CNY     1
EAU     1
VEL     1
PBG     1
TOL     1
Name: count, dtype: int64

Between June 2003 and December 2024, Adak Airport (ADK) recorded the highest flight delay rates, topping the ranking 21 times. It was followed by Pago Pago International Airport (PPG) and Southwest Oregon Regional Airport (OTH), each leading 19 times.

Notably, Adak Airport (ADK) also recorded the highest cancellation rates, leading the ranking 32 times. It was followed by Mammoth Yosemite Airport (MMH) (19 times) and Kodiak Airport (ADQ) (18 times).

For flight diversions, Friedman Memorial Airport (SUN) ranked highest 54 times, far surpassing other airports. It was followed by Ogden-Hinckley Airport (OGD) and Southwest Oregon Regional Airport (OTH), each topping the ranking 11 times.

### 3.5. Disruptions by delay cause

#### *3.5.1. Long-term trends in delay causes*

We can categorise the monthly reported flight delays by their causes and visualise their proportions using a stacked bar plot, covering the period from June 2003 to December 2024.

In [85]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Group by month and sum the total count of delays and delay causes
monthly_delay_data = df.groupby('year_month', observed=True).agg({
    'arr_del15': 'sum',         # Total delays by month
    'carrier_ct': 'sum',        # Total delays caused by carrier
    'weather_ct': 'sum',        # Total delays caused by weather
    'nas_ct': 'sum',            # Total delays caused by NAS
    'security_ct': 'sum',       # Total delays caused by security
    'late_aircraft_ct': 'sum'   # Total delays caused by late aircraft
}).reset_index()

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Compute the proportion of each delay cause by dividing by total delays (arr_del15)
monthly_delay_data['carrier_pct'] = monthly_delay_data['carrier_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['weather_pct'] = monthly_delay_data['weather_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['nas_pct'] = monthly_delay_data['nas_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['security_pct'] = monthly_delay_data['security_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['late_aircraft_pct'] = monthly_delay_data['late_aircraft_ct'] / monthly_delay_data['arr_del15']

# Reshape the DataFrame to have one row per month and cause for easier plotting
plot_data = monthly_delay_data.melt(
    id_vars='year_month', 
    value_vars=['carrier_pct', 'weather_pct', 'nas_pct', 'security_pct', 'late_aircraft_pct'], 
    var_name='delay_cause', 
    value_name='proportion'
)

# Rename delay causes for better legend labels
plot_data['delay_cause'] = plot_data['delay_cause'].replace({
    'carrier_pct': 'Carrier Delay',
    'weather_pct': 'Weather Delay',
    'nas_pct': 'NAS Delay',
    'security_pct': 'Security Delay',
    'late_aircraft_pct': 'Late Aircraft Delay'
})

# Set up DatetimeTickFormatter for customized tick labels
tickfmt = DatetimeTickFormatter(years="%Y-%m", months="%Y-%m")

# Set up tooltips for interactivity
tooltips = [
    ("Year-Month", "@year_month{%Y-%m}"),  # Formatting date as Year-Month
    ("Delay Cause", "@delay_cause"),
    ("proportion", "@proportion{0.000}")
]
hover = HoverTool(tooltips=tooltips, formatters={"@year_month": "datetime"})

# Plot using hvplot with a stacked bar chart and show_legend=False
bar_plot = plot_data.hvplot.bar(
    x='year_month', 
    y='proportion', 
    by='delay_cause',  # Group bars by delay cause
    stacked=True,      # Stacked bars
    xformatter=tickfmt,
    tools=[hover],
    title='Figure 8: Proportion of delay causes by year-month', 
    xlabel='Year-Month', 
    ylabel='Proportion of Delays',
    line_width=0
).opts(width=1000, height=500, show_legend=False)

display(bar_plot)

#---------------------------------------------------------------------------------------------------------------------------------------------------
# LEGEND

# Create a DataFrame with 5 columns and 1 row
data = {'Legend': ['NAS Delay', 'Security Delay', 'Weather Delay', 'Late Aircraft Delay', 'Carrier Delay']}

table = pd.DataFrame(data)

# Define a color map for each cell
cell_colors = ['#878787', '#6d904f', '#e4ad37', '#fc4f30', '#30a2da']

# Apply the background color to each cell
def apply_colors(val):
    color = cell_colors.pop(0)
    return f'background-color: {color}'

# Apply colors to the table
styled_table = table.style.map(apply_colors)

# Hide the index
styled_table = styled_table.hide(axis="index")

# Display the styled table
display(styled_table)

:Bars   [year_month,delay_cause]   (proportion)

Legend
NAS Delay
Security Delay
Weather Delay
Late Aircraft Delay
Carrier Delay


From Figure 8, we can see that the most common causes of flight delays over the entire time frame are National Airspace System (NAS) delays, late-arriving aircraft, and carrier-related issues. Weather-related delays follow at a distant fourth, with security delays being the least frequent.

Similar to our findings in previous sections, April 2020 stands out as an anomalous month for delay causes. During this period, carrier delays rose to account for 50% of all flight delays, while late-aircraft delays dropped to just 15%. The COVID-19 pandemic might have caused carriers to struggle with staffing shortages and route suspensions, causing carrier delays to spike during this period (Sun *et al.*, 2023).

#### *3.5.2. Seasonality of delay causes*

Next, we analyse the seasonality of delay causes. As with our previous analyses that work average monthly data across multiple years, we exclude the COVID-19 period.

In [88]:
# Data preparation: convert year-month to datetime
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')

# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Convert year-month for df back to string
df['year_month'] = df['year_month'].dt.strftime('%Y-%m')

# Group by month and calculate total delays, and total for each of the 5 delay causes, for each month across all years
monthly_total_metrics = df_filtered.groupby('month').agg({
    'arr_del15': 'sum',         # Total delays per month
    'carrier_ct': 'sum',        # Total delays caused by carrier per month
    'weather_ct': 'sum',        # Total delays caused by weather per month
    'nas_ct': 'sum',            # Total delays caused by NAS per month
    'security_ct': 'sum',       # Total delays caused by security per month
    'late_aircraft_ct': 'sum'   # Total delays caused by late aircraft per month
}).reset_index()

# Count the number of available years for each month (excluding COVID-19 impacted months)
years_per_month = df_filtered.groupby('month')['year'].nunique().reset_index()
years_per_month.columns = ['month', 'available_years']

# Merge total metrics with the count of available years
monthly_delay_data = pd.merge(monthly_total_metrics, years_per_month, on='month')

# Compute the proportion of each delay cause by dividing by total delays (arr_del15)
monthly_delay_data['carrier_pct'] = monthly_delay_data['carrier_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['weather_pct'] = monthly_delay_data['weather_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['nas_pct'] = monthly_delay_data['nas_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['security_pct'] = monthly_delay_data['security_ct'] / monthly_delay_data['arr_del15']
monthly_delay_data['late_aircraft_pct'] = monthly_delay_data['late_aircraft_ct'] / monthly_delay_data['arr_del15']

# Reshape the DataFrame to have one row per month and cause for easier plotting
monthly_delay_data = monthly_delay_data.melt(
    id_vars='month', 
    value_vars=['carrier_pct', 'weather_pct', 'nas_pct', 'security_pct', 'late_aircraft_pct'], 
    var_name='delay_cause', 
    value_name='proportion'
)

# Rename delay causes for better legend labels
monthly_delay_data['delay_cause'] = monthly_delay_data['delay_cause'].replace({
    'carrier_pct': 'Carrier Delay',
    'weather_pct': 'Weather Delay',
    'nas_pct': 'NAS Delay',
    'security_pct': 'Security Delay',
    'late_aircraft_pct': 'Late Aircraft Delay'
})

# Set the order of delay causes for the stacked bar plot
delay_cause_order = ['Carrier Delay', 'Late Aircraft Delay', 'Weather Delay', 'Security Delay', 'NAS Delay']
monthly_delay_data['delay_cause'] = pd.Categorical(monthly_delay_data['delay_cause'], categories=delay_cause_order, ordered=True)

# Ensure the DataFrame is sorted by the order of delay causes before plotting
monthly_delay_data = monthly_delay_data.sort_values('delay_cause')

# Convert month numbers (1-12) to month names
monthly_delay_data['Month Name'] = monthly_delay_data['month'].apply(lambda x: calendar.month_abbr[x])

# Set up tooltips for interactivity
tooltips = [
    ("Delay Cause", "@delay_cause"),
    ("Proportion", "@proportion{0.000}")
]
hover = HoverTool(tooltips=tooltips)

# Plot using hvplot with a stacked bar chart and show_legend=False
bar_plot = monthly_delay_data.hvplot.bar(
    x='Month Name', 
    y='proportion', 
    by='delay_cause',  # Group bars by delay cause
    stacked=True,      # Stacked bars
    tools=[hover],
    title='Figure 9: Proportion of delay causes from January to December \n(excluding COVID-19 period)', 
    xlabel='Month', 
    ylabel='Proportion of delay causes',
    line_width=0
).opts(width=1000, height=500, show_legend=False)

display(bar_plot)

#---------------------------------------------------------------------------------------------------------------------------------------------------
# LEGEND

# Create a DataFrame with 5 columns and 1 row
data = {'Legend': ['NAS Delay', 'Security Delay', 'Weather Delay', 'Late Aircraft Delay', 'Carrier Delay']}

table = pd.DataFrame(data)

# Define a color map for each cell
cell_colors = ['#878787', '#6d904f', '#e4ad37', '#fc4f30', '#30a2da']

# Apply the background color to each cell
def apply_colors(val):
    color = cell_colors.pop(0)
    return f'background-color: {color}'

# Apply colors to the table
styled_table = table.style.map(apply_colors)

# Hide the index
styled_table = styled_table.hide(axis="index")

# Display the styled table
display(styled_table)

:Bars   [Month Name,delay_cause]   (proportion)

Legend
NAS Delay
Security Delay
Weather Delay
Late Aircraft Delay
Carrier Delay


The findings from Figure 9 closely mirror those of Figure 8, with the most and least common delay causes remaining largely consistent across different months.

## 4. Geospatial analysis

### 4.1. Disruptions by scenario

We start by exploring the geographic distribution of airports by their rate of delays, cancellations, and diversions. Similar to our previous analysis on seasonality, we exclude the monthly data collected during the COVID-19 pandemic as they are not representative of usual flight operations.

In [92]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_flights': 'sum',
    'arr_del15': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate delay rate
grouped['delay_rate'] = grouped['arr_del15'] / grouped['arr_flights']

# Fill NaN values in delay_rate with 0
grouped['delay_rate'] = grouped['delay_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest delay, 1 = lowest delay)
grouped['delay_rate_quantile'] = pd.qcut(grouped['delay_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_flights': 'Number of arriving flights',
    'arr_del15': 'Number of delayed flights',
    'delay_rate': 'Delay Rate',
    'delay_rate_quantile': 'Delay Rate Quantile'
})

# Format columns
grouped['Number of arriving flights'] = grouped['Number of arriving flights'].apply(lambda x: f"{x:,}")
grouped['Number of delayed flights'] = grouped['Number of delayed flights'].apply(lambda x: f"{x:,}")
grouped['Delay Rate'] = grouped['Delay Rate'].apply(lambda x: f"{x:.3f}")


# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 10: Airports ranked by delay rate in the United States \n(excluding COVID-19 period)",
    color='Delay Rate Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of arriving flights', 
        'Number of delayed flights', 
        'Delay Rate', 
        'Delay Rate Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Delay Rate Quantile,Airport,Number of arriving flights,Number of delayed flights,Delay Rate)

Figure 10 suggests that airports with higher delay rates tend to be concentrated in the eastern half of the contiguous U.S. Notably, airports in the New York metropolitan area and Florida rank among the highest 10% for delay rates. Other airports with among the highest delay rates can be found in certain parts of California, the Aleutian Islands, Puerto Rico, and American Samoa.

Airports that rank among the lowest 10% for delay rates appear to be clustered in Hawaii, Guam, Southern California and the Mountain states of the western U.S.

In [94]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_flights': 'sum',
    'arr_cancelled': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate cancellation rate
grouped['cancellation_rate'] = grouped['arr_cancelled'] / grouped['arr_flights']

# Fill NaN values in cancellation_rate with 0
grouped['cancellation_rate'] = grouped['cancellation_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest cancel rate, 1 = lowest cancel rate)
grouped['cancellation_rate_quantile'] = pd.qcut(grouped['cancellation_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_flights': 'Number of arriving flights',
    'arr_cancelled': 'Number of cancelled flights',
    'cancellation_rate': 'Cancellation Rate',
    'cancellation_rate_quantile': 'Cancellation Rate Quantile'
})

# Format columns
grouped['Number of arriving flights'] = grouped['Number of arriving flights'].apply(lambda x: f"{x:,}")
grouped['Number of cancelled flights'] = grouped['Number of cancelled flights'].apply(lambda x: f"{x:,}")
grouped['Cancellation Rate'] = grouped['Cancellation Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 11: Airports ranked by cancellation rate in the United States \n(excluding COVID-19 period)",
    color='Cancellation Rate Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of arriving flights', 
        'Number of cancelled flights', 
        'Cancellation Rate', 
        'Cancellation Rate Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Cancellation Rate Quantile,Airport,Number of arriving flights,Number of cancelled flights,Cancellation Rate)

Airports among the highest 10% for cancellation rates appear to be concentrated in the Midwest, Northeast, and the remote western regions of Alaska. 

Similar to the delay patterns observed in Figure 10, airports in the western U.S. tend to have lower cancellation rates. However, in contrast to Figure 10, airports in Florida, Puerto Rico, and American Samoa have relatively low cancellation rates. In fact, all airports in the outlying U.S. territories perform well in terms of cancellations.

In [96]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_flights': 'sum',
    'arr_diverted': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate diversion rate
grouped['diversion_rate'] = grouped['arr_diverted'] / grouped['arr_flights']

# Fill NaN values in diversion_rate with 0
grouped['diversion_rate'] = grouped['diversion_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest cancel rate, 1 = lowest cancel rate)
grouped['diversion_rate_quantile'] = pd.qcut(grouped['diversion_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_flights': 'Number of arriving flights',
    'arr_diverted': 'Number of diverted flights',
    'diversion_rate': 'Diversion Rate',
    'diversion_rate_quantile': 'Diversion Rate Quantile'
})

# Format columns
grouped['Number of arriving flights'] = grouped['Number of arriving flights'].apply(lambda x: f"{x:,}")
grouped['Number of diverted flights'] = grouped['Number of diverted flights'].apply(lambda x: f"{x:,}")
grouped['Diversion Rate'] = grouped['Diversion Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 12: Airports ranked by diversion rate in the United States \n(excluding COVID-19 period)",
    color='Diversion Rate Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of arriving flights', 
        'Number of diverted flights', 
        'Diversion Rate', 
        'Diversion Rate Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Diversion Rate Quantile,Airport,Number of arriving flights,Number of diverted flights,Diversion Rate)

The geographical distribution of airports based on diversion rates appears to be the opposite of the patterns seen for delays and cancellations. Airports in the Mountain states tend to have higher diversion rates. Additionally, most airports in Alaska perform poorly in this regard.

In contrast, there is a concentration of airports on the West Coast and eastern half of the U.S. that experience lower diversion rates.

### 4.2. Disruptions by delay cause

We now evaluate the geographical distribution of delay causes.

In [99]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_del15': 'sum',
    'nas_ct': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate NAS delay rate
grouped['nas_rate'] = grouped['nas_ct'] / grouped['arr_del15']

# Fill NaN values in nas_rate with 0
grouped['nas_rate'] = grouped['nas_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest delay rate, 1 = lowest delay rate)
grouped['nas_delay_quantile'] = pd.qcut(grouped['nas_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_del15': 'Number of delayed flights',
    'nas_ct': 'Number of delayed flights caused by NAS',
    'nas_rate': 'NAS Delay Rate',
    'nas_delay_quantile': 'NAS Delay Quantile'
})

# Format columns
grouped['Number of delayed flights'] = grouped['Number of delayed flights'].apply(lambda x: f"{x:,}")
grouped['Number of delayed flights caused by NAS'] = grouped['Number of delayed flights caused by NAS'].apply(lambda x: f"{round(x):,}")
grouped['NAS Delay Rate'] = grouped['NAS Delay Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 13: Airports ranked by proportion of delays caused by NAS \n(excluding COVID-19 period)",
    color='NAS Delay Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of delayed flights', 
        'Number of delayed flights caused by NAS', 
        'NAS Delay Rate', 
        'NAS Delay Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (NAS Delay Quantile,Airport,Number of delayed flights,Number of delayed flights caused by NAS,NAS Delay Rate)

Airports along the U.S. East Coast rank among the highest 10% in terms of the proportion of delays attributed to the National Airspace System (NAS). This is likely due to the region's high air traffic volume and congested air space (Sismanidou *et al.*, 2022).

Conversely, airports along the West Coast rank among the lowest 10% for NAS-related delays. This may be due to a combination of less congested airspace and fewer air traffic management interventions.

In [101]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_del15': 'sum',
    'security_ct': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate security delay rate
grouped['security_rate'] = grouped['security_ct'] / grouped['arr_del15']

# Fill NaN values in security_rate with 0
grouped['security_rate'] = grouped['security_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest delay rate, 1 = lowest delay rate)
grouped['security_delay_quantile'] = pd.qcut(grouped['security_rate'], 10, labels=False, duplicates='drop') + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_del15': 'Number of delayed flights',
    'security_ct': 'Number of delayed flights caused by security',
    'security_rate': 'Security Delay Rate',
    'security_delay_quantile': 'Security Delay Quantile'
})

# Format columns
grouped['Number of delayed flights'] = grouped['Number of delayed flights'].apply(lambda x: f"{x:,}")
grouped['Number of delayed flights caused by security'] = grouped['Number of delayed flights caused by security'].apply(lambda x: f"{round(x):,}")
grouped['Security Delay Rate'] = grouped['Security Delay Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 14: Airports ranked by proportion of delays caused by security \n(excluding COVID-19 period)",
    color='Security Delay Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of delayed flights', 
        'Number of delayed flights caused by security', 
        'Security Delay Rate', 
        'Security Delay Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Security Delay Quantile,Airport,Number of delayed flights,Number of delayed flights caused by security,Security Delay Rate)

Airports in Alaska, Hawaii, American Samoa, Puerto Rico, and along the West Coast rank among the highest 10% for security-related delays. This suggests that outlying airports may be subject to stricter security checks or additional screening procedures.

In contrast, airports with a lower proportion of security-related delays are primarily located in the eastern half of the U.S, potentially indicating more streamlined security measures.

In [103]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_del15': 'sum',
    'weather_ct': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate weather delay rate
grouped['weather_rate'] = grouped['weather_ct'] / grouped['arr_del15']

# Fill NaN values in weather_rate with 0
grouped['weather_rate'] = grouped['weather_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest delay rate, 1 = lowest delay rate)
grouped['weather_delay_quantile'] = pd.qcut(grouped['weather_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_del15': 'Number of delayed flights',
    'weather_ct': 'Number of delayed flights caused by weather',
    'weather_rate': 'Weather Delay Rate',
    'weather_delay_quantile': 'Weather Delay Quantile'
})

# Format columns
grouped['Number of delayed flights'] = grouped['Number of delayed flights'].apply(lambda x: f"{x:,}")
grouped['Number of delayed flights caused by weather'] = grouped['Number of delayed flights caused by weather'].apply(lambda x: f"{round(x):,}")
grouped['Weather Delay Rate'] = grouped['Weather Delay Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 15: Airports ranked by proportion of delays caused by weather \n(excluding COVID-19 period)",
    color='Weather Delay Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of delayed flights', 
        'Number of delayed flights caused by weather', 
        'Weather Delay Rate', 
        'Weather Delay Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Weather Delay Quantile,Airport,Number of delayed flights,Number of delayed flights caused by weather,Weather Delay Rate)

Airports with a higher proportion of weather-related delays are primarily located in Alaska and the eastern half of the U.S.. This could be due to the occurrence of more severe winter storms, hurricanes, and other adverse weather conditions in these regions (Bombelli and Sallan, 2023).

Conversely, airports with the lowest 10% of weather-related delays are overwhelmingly found along the West Coast and in Hawaii, where milder and more predictable weather conditions contribute to fewer disruptions.

In [105]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_del15': 'sum',
    'late_aircraft_ct': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate late aircraft delay rate
grouped['late_aircraft_rate'] = grouped['late_aircraft_ct'] / grouped['arr_del15']

# Fill NaN values in late_aircraft_rate with 0
grouped['late_aircraft_rate'] = grouped['late_aircraft_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest delay rate, 1 = lowest delay rate)
grouped['late_aircraft_delay_quantile'] = pd.qcut(grouped['late_aircraft_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_del15': 'Number of delayed flights',
    'late_aircraft_ct': 'Number of delayed flights caused by late aircraft',
    'late_aircraft_rate': 'Late Aircraft Delay Rate',
    'late_aircraft_delay_quantile': 'Late Aircraft Delay Quantile'
})

# Format columns
grouped['Number of delayed flights'] = grouped['Number of delayed flights'].apply(lambda x: f"{x:,}")
grouped['Number of delayed flights caused by late aircraft'] = grouped['Number of delayed flights caused by late aircraft'].apply(lambda x: f"{round(x):,}")
grouped['Late Aircraft Delay Rate'] = grouped['Late Aircraft Delay Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 16: Airports ranked by proportion of delays caused by late aircraft \n(excluding COVID-19 period)",
    color='Late Aircraft Delay Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of delayed flights', 
        'Number of delayed flights caused by late aircraft', 
        'Late Aircraft Delay Rate', 
        'Late Aircraft Delay Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Late Aircraft Delay Quantile,Airport,Number of delayed flights,Number of delayed flights caused by late aircraft,Late Aircraft Delay Rate)

Airports that rank among the top 10% in terms of delays caused by late aircraft are overwhelmingly concentrated along the West Coast and Southeast Alaska. This may be due to the longer flight routes these airports handle compared to their East Coast counterparts, leading to greater susceptibility to cascading delays from inbound flights.

In comparison, airports in the Mountain states and the Lower South perform are among the airports least affected by delays caused by late aircraft. This suggests more efficient scheduling, lower congestion, or fewer cascading delays from inbound flights.

In [107]:
# Exclude data from April 2020 to May 2021 due to the COVID-19 impact
df_filtered = df[(df['year_month'] < '2020-04') | (df['year_month'] > '2021-05')]

# Group by airport and calculate sums
grouped = df_filtered.groupby('airport', as_index=False).agg({
    'arr_del15': 'sum',
    'carrier_ct': 'sum',
    'lat': 'first',  # Keep one lat/lon per airport
    'lon': 'first'
})

# Calculate carrier delay rate
grouped['carrier_rate'] = grouped['carrier_ct'] / grouped['arr_del15']

# Fill NaN values in carrier_rate with 0
grouped['carrier_rate'] = grouped['carrier_rate'].fillna(0)

# Assign quantile-based ranks (10 = highest delay rate, 1 = lowest delay rate)
grouped['carrier_delay_quantile'] = pd.qcut(grouped['carrier_rate'], 10, labels=False) + 1

# Rename columns for better hover display
grouped = grouped.rename(columns={
    'lon': 'Longitude',
    'lat': 'Latitude',
    'airport': 'Airport',
    'arr_del15': 'Number of delayed flights',
    'carrier_ct': 'Number of delayed flights caused by carrier',
    'carrier_rate': 'Carrier Delay Rate',
    'carrier_delay_quantile': 'Carrier Delay Quantile'
})

# Format columns
grouped['Number of delayed flights'] = grouped['Number of delayed flights'].apply(lambda x: f"{x:,}")
grouped['Number of delayed flights caused by carrier'] = grouped['Number of delayed flights caused by carrier'].apply(lambda x: f"{round(x):,}")
grouped['Carrier Delay Rate'] = grouped['Carrier Delay Rate'].apply(lambda x: f"{x:.3f}")

# Plot the geographic distribution of airports
grouped.hvplot(
    geo=True,                  
    x='Longitude',             
    y='Latitude',              
    kind='points',             
    tiles='OSM',               
    title="Figure 17: Airports ranked by proportion of delays caused by carrier \n(excluding COVID-19 period)",
    color='Carrier Delay Quantile',
    cmap='RdYlGn_r',  # Reverse colormap
    size=10,                     
    xlim=(-170, -60),           
    ylim=(15, 72),
    hover_cols=[
        'Airport', 
        'Number of delayed flights', 
        'Number of delayed flights caused by carrier', 
        'Carrier Delay Rate', 
        'Carrier Delay Quantile']
).opts(frame_width=700, frame_height=400)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (Carrier Delay Quantile,Airport,Number of delayed flights,Number of delayed flights caused by carrier,Carrier Delay Rate)

Airports with a higher proportion of carrier-related delays are primarily concentrated in the western half of the U.S. and in outlying U.S. territories. This may be due to operational challenges faced by smaller or less-connected airports, such as limited resources, staffing constraints.

Conversely, the best-performing airports in terms of carrier-related delays tend to be major hubs in large cities, such as those in the New York metropolitan area. These hubs likely benefit from greater infrastructure and more efficient scheduling.

## 5. Conclusion

### 5.1. Summary

This research has highlighted the temporal and geospatial variations in domestic flights across the United States. The findings from the temporal analysis align closely with existing literature, particularly regarding the significant impacts of the Great Recession and the COVID-19 pandemic on air travel. Additionally, the analysis reaffirms the well-established seasonal patterns in air travel, with peak travel months occurring during the summer. 

From a geospatial perspective, the study highlights distinct regional differences in disruption rates and delay causes, especially between the eastern and western halves of the country. These findings provide valuable insights into the operational challenges faced by different regions, thereby informing future strategies for improving air travel efficiency.

### 5.2. Suggestions

For the sake of brevity, temporal and geospatial analyses were conducted separately for this research. A future direction would be to integrate these approaches to explore how flight disruptions evolve across both time and location, offering a more comprehensive understanding of patterns and trends in air travel.

Future research could also account for the magnitude of flight disruptions by integrating the total travel time lost due to delays, cancellations, and diversions. This would provide a more nuanced assessment of the extent and severity of flight disruptions.

## References

Atwood, K., Zeleny, J., Salama, V., Diamond, J., Sands, G. and Wallace, G. (2020). 'US domestic air travel sees ‘virtual shutdown’ as more restrictions are being discussed', *CNN*, 19 March. Available at: https://edition.cnn.com/2020/03/19/politics/domestic-travel-restrictions-virtual-shutdown/index.html (Accessed: 20 March 2025).

Bombelli, A. and Sallan, J.M. (2023). Analysis of the effect of extreme weather on the US domestic air network. A delay and cancellation propagation network approach. *Journal of Transport Geography, 107*, p.103541. https://doi.org/10.1016/j.jtrangeo.2023.103541.

Borsetti, M. (2025). 'airportsdata 20250224', *PyPI*, 24 February. Available at: https://pypi.org/project/airportsdata/ (Accessed: 20 March 2025).

Bureau of Transportation Statistics. (2025). *Airline On-Time Statistics and Delay Causes*. Available at: https://www.transtats.bts.gov/OT_Delay/OT_DelayCause1.asp?20=E (Accessed: 20 March 2025).

Segal, S. (2018). 'Analysis: What the Great Recession meant for aircraft funding', *FlightGlobal*, 7 September. Available at: https://www.flightglobal.com/analysis/analysis-what-the-great-recession-meant-for-aircraft-funding/129424.article (Accessed: 20 March 2025).

Sismanidou, A., Tarradellas, J. and Suau-Sanchez, P. (2022). The uneven geography of US air traffic delays: Quantifying the impact of connecting passengers on delay propagation. *Journal of Transport Geography, 98*, p.103260. https://doi.org/10.1016/j.jtrangeo.2021.103260.

Sun, X., Wandelt, S. and Zhang, A. (2023). A data-driven analysis of the aviation recovery from the COVID-19 pandemic. *Journal of Air Transport Management, 109*, p.102401. https://doi.org/10.1016/j.jairtraman.2023.102401.

## Word count

In [114]:
import io
import os
from nbformat import current
import re

# Initialize word count variables
total_markdown = 0
total_heading = 0
total_code = 0

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".ipynb") and not file.endswith("checkpoint.ipynb"):
            print(os.path.join(root, file))
            with io.open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                nb = current.read(f, 'json')

            word_count_markdown = 0
            word_count_heading = 0
            word_count_code = 0

            for cell in nb.worksheets[0].cells:
                # Check for markdown cells
                if cell.cell_type == "markdown":
                    markdown_content = cell['source']
                    # Split markdown content at 'References', 'Word count', or 'Appendix' sections
                    parts = re.split(r'\bReferences\b|\bWord count\b|\bAppendix\b', markdown_content, flags=re.IGNORECASE)
                    # Only consider the part before the excluded sections for word count
                    if parts:
                        markdown_content_before_exclusions = parts[0]
                        word_count_markdown += len(markdown_content_before_exclusions.replace('#', '').lstrip().split(' '))
                
                # Check for heading cells
                elif cell.cell_type == "heading":
                    heading_content = cell['source'].strip()
                    # Exclude heading if it contains "References", "Word count", or "Appendix"
                    if not re.search(r'\bReferences\b|\bWord count\b|\bAppendix\b', heading_content, flags=re.IGNORECASE):
                        word_count_heading += len(heading_content.replace('#', '').lstrip().split(' '))
                
                # Check for code cells
                elif cell.cell_type == "code":
                    word_count_code += len(cell['input'].replace('#', '').lstrip().split(' '))

            total_markdown += word_count_markdown
            total_heading += word_count_heading
            total_code += word_count_code

            # Output the word counts for this notebook
            print(f"{word_count_markdown} words in notebook's markdown (excluding references and word count sections)")
            print(f"{word_count_heading} words in notebook's headings (excluding references and word count headings)")
            print(f"{word_count_code} words in notebook's code")
            print("(Within 10% of stipulated word limit)")

.\Summative_3061039.ipynb
3299 words in notebook's markdown (excluding references and word count sections)
38 words in notebook's headings (excluding references and word count headings)
7126 words in notebook's code
(Within 10% of stipulated word limit)


C:\Users\ryann\AppData\Local\Temp\ipykernel_13744\1060601777.py:3: DeprecationWarning: nbformat.current is deprecated since before nbformat 3.0

- use nbformat for read/write/validate public API
- use nbformat.vX directly to composing notebooks of a particular version

  from nbformat import current
